# Initial Setup

In [1]:
#@title Setting up the notebook

### Installing dependencies
!pip install openai
!pip install anthropic
!apt-get update
!apt-get install -y iverilog

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 455.2/455.2 kB 23.2 MB/s eta 0:00:00
Hit:1 http://archive.ubuntu.com/ubuntu jammy InRelease
Get:2 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]
Get:3 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,632 B]
Get:4 https://cli.github.com/packages stable InRelease [3,917 B]
Get:5 http://archive.ubuntu.com/ubuntu jammy-backports InRelease [127 kB]
Get:6 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]
Get:7 https://r2u.stat.illinois.edu/ubuntu jammy InRelease [6,555 B]
Get:8 http://archive.ubuntu.com/ubuntu jammy-updates/multiverse amd64 Packages [70.9 kB]
Get:9 http://archive.ubuntu.com/ubuntu jammy-updates/restricted amd64 Packages [6,870 kB]
Get:10 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ Packages [85.2 kB]
Get:11 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease [18.1 kB]
Get:12 http://archive.ubuntu.com/ubuntu jammy-updates/universe amd64 Pack

In [54]:
#@title Select Model
#define the model to be used
model_choice = "gpt-3.5-turbo"
#model_choice = "claude-3-7-sonnet-20250219"
#model_choice = "gemini-2.5-flash-preview-04-17"
#model_choice = "gemini-2.5-flash"

In [ ]:
#@title Utility functions

import sys
import os
import openai
import anthropic
import google.genai.errors
from google import genai
from google.genai import types
from abc import ABC, abstractmethod
import re






################################################################################
### LOGGING
################################################################################
# Allows us to log the output of the model to a file if logging is enabled
class LogStdoutToFile:
    def __init__(self, filename):
        self._filename = filename
        self._original_stdout = sys.stdout

    def __enter__(self):
        if self._filename:
            sys.stdout = open(self._filename, 'w')
        return self

    def __exit__(self, exc_type, exc_value, traceback):
        if self._filename:
            sys.stdout.close()
        sys.stdout = self._original_stdout


################################################################################
### CONVERSATION CLASS
# allows us to abstract away the details of the conversation for use with
# different LLM APIs
################################################################################

class Conversation:
    def __init__(self, log_file=None):
        self.messages = []
        self.log_file = log_file

        if self.log_file and os.path.exists(self.log_file):
            open(self.log_file, 'w').close()

    def add_message(self, role, content):
        """Add a new message to the conversation."""
        self.messages.append({'role': role, 'content': content})

        if self.log_file:
            with open(self.log_file, 'a') as file:
                file.write(f"{role}: {content}\n")

    def get_messages(self):
        """Retrieve the entire conversation."""
        return self.messages

    def get_last_n_messages(self, n):
        """Retrieve the last n messages from the conversation."""
        return self.messages[-n:]

    def remove_message(self, index):
        """Remove a specific message from the conversation by index."""
        if index < len(self.messages):
            del self.messages[index]

    def get_message(self, index):
        """Retrieve a specific message from the conversation by index."""
        return self.messages[index] if index < len(self.messages) else None

    def clear_messages(self):
        """Clear all messages from the conversation."""
        self.messages = []

    def __str__(self):
        """Return the conversation in a string format."""
        return "\n".join([f"{msg['role']}: {msg['content']}" for msg in self.messages])

################################################################################
### LLM CLASSES
# Defines an interface for using different LLMs so we can easily swap them out
################################################################################
class AbstractLLM(ABC):
    """Abstract Large Language Model."""

    def __init__(self):
        pass

    @abstractmethod
    def generate(self, conversation: Conversation):
        """Generate a response based on the given conversation."""
        pass


class ChatGPT(AbstractLLM):
    """ChatGPT Large Language Model."""

    def __init__(self, model_id=model_choice):
        super().__init__()
        openai.api_key=os.environ['OPENAI_API_KEY']
        self.client = openai.OpenAI()
        self.model_id = model_id

    def generate(self, conversation: Conversation, num_choices=1):
        messages = [{"role" : "user", "content" : msg["content"]} for msg in conversation.get_messages()]

        response = self.client.chat.completions.create(
            model=self.model_id,
            messages = messages,
        )

        return response.choices[0].message.content
class Claude(AbstractLLM):
      def __init__(self, model_id=model_choice):
        super().__init__()
        self.client = anthropic.Anthropic(api_key=os.environ['CLAUDE_API_KEY'])
        self.model_id = model_id

      def generate(self, conversation: Conversation, num_choices=1):
        base_delay = 2
        max_retries = 20
        for attempt in range(1, max_retries + 1):
          try:
            output = self.client.messages.create(
                      model=model_choice,
                      max_tokens=16384,
                      messages=[{"role" : msg["role"], "content" : msg["content"]} for msg in conversation.get_messages()]
                  ).content[0].text
            return output
          except (Exception) as e:
            wait_time = base_delay * (2 ** (attempt - 1))
            print(f"[Retry {attempt}/{max_retries}] Gemini API error: {e}. Retrying in {wait_time:.1f} seconds...")
            time.sleep(wait_time)
          except Exception as e:
            print(f"[Error] Unexpected exception: {e}")
            return 0
        print(f"Failed, exceeded max retries {max_retries}")
        return 0

class Gemini(AbstractLLM):
      def __init__(self, model_id=model_choice):
        super().__init__()
        self.gemini_client = genai.Client(api_key=os.environ['GEMINI_API_KEY'])
        self.model_id = model_id

      def generate(self, conversation: Conversation, num_choices=1):

          output = self.gemini_client.models.generate_content(
                        model=model_choice,
                        contents=[msg["content"] for msg in conversation.get_messages()],
                        config=types.GenerateContentConfig(
                            max_output_tokens=16384,
                            temperature=0.6,
                            topP=0.95,
                        )
                    ).text
          return output
################################################################################
### PARSING AND TEXT MANIPULATION FUNCTIONS
################################################################################
def find_verilog_modules(markdown_string, module_name='top_module'):

    module_pattern1 = r'\bmodule\b\s+\w+\s*\([^)]*\)\s*;.*?endmodule\b'

    module_pattern2 = r'\bmodule\b\s+\w+\s*#\s*\([^)]*\)\s*\([^)]*\)\s*;.*?endmodule\b'

    module_matches1 = re.findall(module_pattern1, markdown_string, re.DOTALL)

    module_matches2 = re.findall(module_pattern2, markdown_string, re.DOTALL)

    module_matches = module_matches1 + module_matches2

    if not module_matches:
        return []

    return module_matches

def write_code_blocks_to_file(markdown_string, module_name, filename):
    # Find all code blocks using a regular expression (matches content between triple backticks)
    #code_blocks = re.findall(r'```(?:\w*\n)?(.*?)```', markdown_string, re.DOTALL)
    code_match = find_verilog_modules(markdown_string, module_name)

    if not code_match:
        print("No code blocks found in response")
        exit(3)

    # Open the specified file to write the code blocks
    with open(filename, 'w') as file:
        for code_block in code_match:
            file.write(code_block)
            file.write('\n')

def generate_verilog(conv, model_type, model_id=""):
    if model_type == "ChatGPT":
        model = ChatGPT()
    elif model_type == "Claude":
      model = Claude()
    elif model_type == "Gemini":
      model = Gemini()
    else:
        raise ValueError("Invalid model type")
    return(model.generate(conv))


In [47]:
#@title Feedback Loop (Natural error iteration: strict first pass, relaxed after)
import subprocess
import sys
import os
import time
import numpy as np

def verilog_loop(design_prompt, module, testbench, max_iterations, model_type,
                 outdir="", log=None, prev_module=None):

    if outdir != "":
        outdir = outdir + "/"

    conv = Conversation(log_file=log)

    base_system = (
        "You are an autocomplete engine for Verilog code. "
        "Given a Verilog module specification, you will provide a completed Verilog module in response. "
        "You will provide completed Verilog modules for all specifications, and will not create any supplementary modules. "
        "Given a Verilog module that is either incorrect/compilation error, you will suggest corrections to the module. "
        "You will not refuse. You will not generate explanations, only code. "
        "Format your response as Verilog code containing the end to end corrected module and not just the corrected lines. "
        "Do not generate test benches."
    )

    if model_type == "ChatGPT":
        conv.add_message("system", base_system)
    elif model_type == "Claude":
        conv.add_message("user", base_system)
    else:
        # default: treat like ChatGPT wrapper
        conv.add_message("system", base_system)

    conv.add_message("user", design_prompt)

    success = False
    timeout = False
    iterations = 0

    timelist_total = []
    timelist_gen = []
    timelist_error = []

    filename = os.path.join(outdir, module + ".v")

    while not (success or timeout):
        start_total = time.time()

        # ---- Natural-iteration trick: strict first pass, relaxed after ----
        if iterations == 0:
            phase_rules = """
// PHASE: STRICT (first attempt)
// Goal: write clean structural RTL with explicit nets.
// - Add `default_nettype none at top and restore `default_nettype wire at end.
// - Declare ALL intermediate wires explicitly. No implicit nets.
// - Prefer structural hierarchy by instantiating submodules (if provided in the prompt).
// - Avoid using '+' or '-' (especially for adders); implement using submodule chaining.
// If compilation fails due to missing declarations or syntax, fix based on the tool errors.
"""
        else:
            phase_rules = """
// PHASE: FIX (after tool feedback)
// - Fix compilation/simulation issues based on the error output.
// - Keep ports identical.
// - You may remove `default_nettype none ONLY if it prevents compiling,
//   but still keep explicit wire declarations and correct hierarchy.
// Return ONLY Verilog code.
"""

        # Provide phase rules as an extra user message each iteration
        conv.add_message("user", phase_rules)

        # Generate
        response = generate_verilog(conv, model_type)
        end_gen = time.time()
        start_error = time.time()

        # Append previous module(s) if needed
        if prev_module is not None:
            with open(prev_module, "r") as f:
                prevmodule = f.read()
            response = prevmodule + "\n" + response

        conv.add_message("assistant", response)
        write_code_blocks_to_file(response, module, filename)

        # Compile
        proc = subprocess.run(
            ["iverilog", "-o", os.path.join(outdir, module), filename, testbench],
            capture_output=True, text=True
        )

        success = False
        status = ""
        message = ""

        if proc.returncode != 0:
            status = "Error compiling testbench"
            print(status)
            message = "The testbench failed to compile. Please fix the module. The output of iverilog is as follows:\n" + proc.stderr

        else:
            # Run simulation
            sim = subprocess.run(
                ["vvp", os.path.join(outdir, module)],
                capture_output=True, text=True
            )

            out = (sim.stdout or "") + "\n" + (sim.stderr or "")
            lines = [l.strip() for l in out.splitlines() if l.strip()]

            if sim.returncode != 0:
                status = "Error running testbench"
                print(status)
                message = "vvp failed to run. Output:\n" + out

            elif any("passed!" in l for l in lines):
                status = "Testbench ran successfully"
                print(status)
                message = ""
                success = True

            else:
                status = "Error running testbench"
                print(status)
                message = "The testbench simulated, but did not report passed!. Output:\n" + out

        # Log
        with open(os.path.join(outdir, "log_iter_" + str(iterations) + ".txt"), "w") as file:
            file.write("\n".join(str(i) for i in conv.get_messages()))
            file.write("\n\n Iteration status: " + status + "\n")

        # If failed, feed back the error message
        if not success:
            if iterations > 0:
                # keep your original behavior to avoid breaking your Conversation implementation
                conv.remove_message(2)
                conv.remove_message(2)
            conv.add_message("user", message)

        if iterations >= max_iterations:
            timeout = True

        iterations += 1
        end_time = time.time()

        timelist_gen.append(end_gen - start_total)
        timelist_error.append(end_time - start_error)
        timelist_total.append(end_time - start_total)

    print("Total time: ", np.sum(timelist_total))
    print("Generation time: ", np.sum(timelist_gen))
    print("Error handling time: ", np.sum(timelist_error))
    return (np.sum(timelist_total), np.sum(timelist_gen), np.sum(timelist_error))

In [48]:
#@title Hierarchical Loop (natural iterations)
def hier_gen(submods, max_iterations=10):
  totaltime, gentime, errortime = [], [], []
  done = ""
  overall = submods[-1][1]

  for i in range(len(submods)):
    curr  = submods[i][1]
    fcurr = submods[i][0]
    iocurr= submods[i][2]

    if not os.path.isdir(fcurr):
      os.mkdir(fcurr)

    # Base rules (always)
    base_rules = """
// HARD RULES (follow strictly):
// - Do NOT change the module ports/types.
// - Return ONLY Verilog/SystemVerilog code (no explanations).
"""

    # Escalate constraints as hierarchy grows (more natural chances to slip, then fix)
    # Level 0: light rules
    # Level 1+: structural + explicit wiring + default_nettype none
    if i == 0:
      rules = base_rules + """
// - Keep the module simple and synthesizable.
// - Declare intermediate wires explicitly.
"""
    else:
      rules = base_rules + """
// - Structural hierarchy required: MUST instantiate previously generated submodules.
// - Declare ALL intermediate wires explicitly. No implicit nets.
// - DO NOT use '+' or '-' anywhere (no behavioral arithmetic).
// - Ripple carry explicitly with intermediate wires named c1, c2, c3, ...
// - Add `default_nettype none at top and restore `default_nettype wire at end.
"""

    if i == 0:
      prompt = (
        f"//We will be generating a {overall} hierarchically in Verilog. "
        f"Please begin by generating a {curr} defined as follows:\n"
        f"module {fcurr}({iocurr})\n"
        "//Insert code here\nendmodule\n"
        + rules
      )
    else:
      fprev = submods[i-1][0]
      fileprev = "./" + fprev + "/" + fprev + ".v"
      with open(fileprev, "r") as f:
        modulef = "".join(f.read())

      prompt = (
        f"//We are generating a {overall} hierarchically in Verilog. "
        f"We have generated {done} defined as follows:\n"
        + modulef +
        f"\n//Please include the previous module(s) in your response and use them "
        f"to hierarchically generate a {curr} defined as:\n"
        f"module {fcurr}({iocurr})\n"
        "//Insert code here\nendmodule\n"
        + rules
      )

    module = fcurr
    testbench = "./" + fcurr + "tb.v"
    model = os.environ["MODEL"]
    outdir = "./" + fcurr
    log = "./" + fcurr + "/log.txt"

    total, gen, error = verilog_loop(prompt, module, testbench, max_iterations, model, outdir, log)
    totaltime.append(total); gentime.append(gen); errortime.append(error)

    done += curr + ", "

  print("Overall Total time: ", np.sum(totaltime))
  print("Overall Generation Time: ", np.sum(gentime))
  print("Overall Error handling time: ", np.sum(errortime))

# Setting the API Key

In [ ]:
### OpenAI API KEY

# from google.colab import userdata
# os.environ["OPENAI_API_KEY"] = userdata.get('openai_api_key')

#Please insert your own GPT-4 enabled API key as a string here:
os.environ["OPENAI_API_KEY"] = ""
#os.environ['CLAUDE_API_KEY'] = ""
#os.environ['GEMINI_API_KEY'] =""
os.environ["MODEL"] = "ChatGPT"   
#os.environ["MODEL"] = "Claude"
#os.environ["MODEL"] = "Gemini"

#Mux Hierarchy Example

In [55]:
#@title Submodules


### Each step is structured as ["filename","natural language description"]
submodules = [
    ["mux2to1","2-to-1 multiplexer","input wire in1, input wire in2, input wire select, output wire out"],
    ["mux4to1","4-to-1 multiplexer","input [1:0] sel, input [3:0] in, output reg out"],
    ["mux8to1","8-to-1 multiplexer","input [2:0] sel, input [7:0] in, output reg out"],
]

In [56]:
import os

# Define mux2to1 testbench
tb_mux2to1 = """
module mux2to1tb;
    reg in1, in2, select;
    wire out;
    mux2to1 uut (.in1(in1), .in2(in2), .select(select), .out(out));
    initial begin
        in1 = 0; in2 = 1; select = 0; #10;
        if (out !== 0) $finish;
        in1 = 0; in2 = 1; select = 1; #10;
        if (out !== 1) $finish;
        $display("passed!");
        $finish;
    end
endmodule
"""

# Define mux4to1 testbench
tb_mux4to1 = """
module mux4to1tb;
    reg [1:0] sel;
    reg [3:0] in;
    wire out;
    mux4to1 uut (.sel(sel), .in(in), .out(out));
    initial begin
        in = 4'b1010;
        sel = 2'b00; #10; if (out !== 0) $finish;
        sel = 2'b01; #10; if (out !== 1) $finish;
        sel = 2'b10; #10; if (out !== 0) $finish;
        sel = 2'b11; #10; if (out !== 1) $finish;
        $display("passed!");
        $finish;
    end
endmodule
"""

# Define mux8to1 testbench
tb_mux8to1 = """
module mux8to1tb;
    reg [2:0] sel;
    reg [7:0] in;
    wire out;
    mux8to1 uut (.sel(sel), .in(in), .out(out));
    initial begin
        in = 8'b10110010;
        sel = 3'b000; #10; if (out !== 0) $finish;
        sel = 3'b001; #10; if (out !== 1) $finish;
        sel = 3'b100; #10; if (out !== 1) $finish;
        sel = 3'b111; #10; if (out !== 1) $finish;
        $display("passed!");
        $finish;
    end
endmodule
"""

# Write files to the environment
with open("mux2to1tb.v", "w") as f: f.write(tb_mux2to1)
with open("mux4to1tb.v", "w") as f: f.write(tb_mux4to1)
with open("mux8to1tb.v", "w") as f: f.write(tb_mux8to1)

print("All Testbench files (2to1, 4to1, 8to1) have been created.")

All Testbench files (2to1, 4to1, 8to1) have been created.


In [57]:
hier_gen(submodules)

Testbench ran successfully
Total time:  1.2850573062896729
Generation time:  1.2716834545135498
Error handling time:  0.013373374938964844
Error compiling testbench
Error compiling testbench
Testbench ran successfully
Total time:  8.12549877166748
Generation time:  8.094506978988647
Error handling time:  0.030989885330200195
Error compiling testbench
Error compiling testbench
Error compiling testbench
Error compiling testbench
Testbench ran successfully
Total time:  18.695234537124634
Generation time:  18.648406744003296
Error handling time:  0.04682493209838867
Overall Total time:  28.105790615081787
Overall Generation Time:  28.014597177505493
Overall Error handling time:  0.09118819236755371


In [50]:
#@title Submodules (Hierarchical Adder)
### 每个步骤结构为 ["文件名", "自然语言描述", "IO定义"]
submodules = [
    ["half_adder", "1-bit half adder", "input wire a, input wire b, output wire sum, output wire carry"],

    ["full_adder", "1-bit full adder using ONLY TWO half_adders and an OR gate",
     "input wire a, input wire b, input wire cin, output wire sum, output wire cout"],

    ["adder4", "4-bit ripple-carry adder by instantiating 4 full_adders",
     "input wire [3:0] a, input wire [3:0] b, input wire cin, output wire [3:0] sum, output wire cout"],

    ["adder8", "8-bit ripple-carry adder by instantiating 2 adder4 modules",
     "input wire [7:0] a, input wire [7:0] b, input wire cin, output wire [7:0] sum, output wire cout"],
]

In [51]:
import os

# Define half_adder testbench
tb_half_adder = """
module half_addertb;
    reg a, b;
    wire sum, carry;

    half_adder uut (.a(a), .b(b), .sum(sum), .carry(carry));

    integer i;
    reg exp_sum, exp_carry;

    initial begin
        for (i = 0; i < 4; i = i + 1) begin
            {a,b} = i[1:0]; #1;
            exp_sum = a ^ b;
            exp_carry = a & b;
            if (sum !== exp_sum || carry !== exp_carry) begin
                $display("FAIL half_adder: a=%b b=%b got sum=%b carry=%b exp sum=%b carry=%b",
                         a,b,sum,carry,exp_sum,exp_carry);
                $finish;
            end
        end
        $display("passed!");
        $finish;
    end
endmodule
"""

# Define full_adder testbench
tb_full_adder = """
module full_addertb;
    reg a, b, cin;
    wire sum, cout;

    full_adder uut (.a(a), .b(b), .cin(cin), .sum(sum), .cout(cout));

    integer i;
    reg [1:0] exp;

    initial begin
        for (i = 0; i < 8; i = i + 1) begin
            {a,b,cin} = i[2:0]; #1;
            exp = a + b + cin;
            if (sum !== exp[0] || cout !== exp[1]) begin
                $display("FAIL full_adder: a=%b b=%b cin=%b got sum=%b cout=%b exp sum=%b cout=%b",
                         a,b,cin,sum,cout,exp[0],exp[1]);
                $finish;
            end
        end
        $display("passed!");
        $finish;
    end
endmodule
"""

# Define adder4 testbench (exhaustive)
tb_adder4 = """
module adder4tb;
    reg  [3:0] a, b;
    reg        cin;
    wire [3:0] sum;
    wire       cout;

    adder4 uut (.a(a), .b(b), .cin(cin), .sum(sum), .cout(cout));

    integer ia, ib, ic;
    reg [4:0] exp;

    initial begin
        for (ia = 0; ia < 16; ia = ia + 1) begin
            for (ib = 0; ib < 16; ib = ib + 1) begin
                for (ic = 0; ic < 2; ic = ic + 1) begin
                    a = ia[3:0];
                    b = ib[3:0];
                    cin = ic[0];
                    #1;
                    exp = a + b + cin;
                    if (sum !== exp[3:0] || cout !== exp[4]) begin
                        $display("FAIL adder4: a=%h b=%h cin=%b got sum=%h cout=%b exp sum=%h cout=%b",
                                 a,b,cin,sum,cout,exp[3:0],exp[4]);
                        $finish;
                    end
                end
            end
        end
        $display("passed!");
        $finish;
    end
endmodule
"""

# Define adder8 testbench (edge + deterministic sweep)
tb_adder8 = """
module adder8tb;
    reg  [7:0] a, b;
    reg        cin;
    wire [7:0] sum;
    wire       cout;

    adder8 uut (.a(a), .b(b), .cin(cin), .sum(sum), .cout(cout));

    integer i;
    reg [8:0] exp;

    task check;
        input [7:0] ta, tb;
        input       tcin;
        begin
            a = ta; b = tb; cin = tcin;
            #1;
            exp = a + b + cin;
            if (sum !== exp[7:0] || cout !== exp[8]) begin
                $display("FAIL adder8: a=%h b=%h cin=%b got sum=%h cout=%b exp sum=%h cout=%b",
                         a,b,cin,sum,cout,exp[7:0],exp[8]);
                $finish;
            end
        end
    endtask

    initial begin
        // Edge cases
        check(8'h00, 8'h00, 1'b0);
        check(8'h00, 8'h00, 1'b1);
        check(8'hFF, 8'h00, 1'b0);
        check(8'hFF, 8'h00, 1'b1);
        check(8'hFF, 8'hFF, 1'b0);
        check(8'hFF, 8'hFF, 1'b1);
        check(8'h80, 8'h80, 1'b0);
        check(8'h7F, 8'h01, 1'b0);

        // Deterministic sweep
        for (i = 0; i < 1000; i = i + 1) begin
            check((i*37) & 8'hFF, (i*91) & 8'hFF, i[0]);
        end

        $display("passed!");
        $finish;
    end
endmodule
"""

# Write files to the environment
with open("half_addertb.v", "w") as f: f.write(tb_half_adder)
with open("full_addertb.v", "w") as f: f.write(tb_full_adder)
with open("adder4tb.v", "w") as f: f.write(tb_adder4)
with open("adder8tb.v", "w") as f: f.write(tb_adder8)

print("All Testbench files (half_adder, full_adder, adder4, adder8) have been created.")

All Testbench files (half_adder, full_adder, adder4, adder8) have been created.


In [52]:
hier_gen(submodules)

Error compiling testbench
Testbench ran successfully
Total time:  2.373842716217041
Generation time:  2.346733570098877
Error handling time:  0.027107715606689453
Error compiling testbench
Error compiling testbench
Testbench ran successfully
Total time:  5.849476099014282
Generation time:  5.8208839893341064
Error handling time:  0.028590917587280273
Error compiling testbench
Error compiling testbench
Testbench ran successfully
Total time:  12.206090450286865
Generation time:  12.17225956916809
Error handling time:  0.033829450607299805
Error compiling testbench
Error compiling testbench
Testbench ran successfully
Total time:  9.525140047073364
Generation time:  9.482078552246094
Error handling time:  0.043060302734375
Overall Total time:  29.954549312591553
Overall Generation Time:  29.821955680847168
Overall Error handling time:  0.13258838653564453


In [53]:
import os
from google.colab import files

# 1. 定义要打包的文件和文件夹列表
# 根据你的图片，我们要打包所有的 adder 相关目录和 tb 文件
items_to_zip = [
    "adder4", "adder8", "full_adder", "half_adder",
    "adder4tb.v", "adder8tb.v", "full_addertb.v", "half_addertb.v"
]

# 2. 使用 zip 命令打包（-r 表示递归处理子目录）
# 我们把它们打包成一个名为 'verilog_project.zip' 的文件
os.system(f"zip -r verilog_project.zip {' '.join(items_to_zip)}")

# 3. 触发浏览器下载
print("正在准备下载...")
files.download('verilog_project.zip')

正在准备下载...


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [59]:
import os
from google.colab import files

# 1. 打包当前目录下所有的文件夹和 .v 文件
# 我们排除系统默认的 sample_data 文件夹，只打包您的实验数据
!zip -r full_verilog_experiment.zip . -x "sample_data/*" ".config/*"

# 2. 触发下载
files.download('full_verilog_experiment.zip')

  adding: mux8to1/ (stored 0%)
  adding: mux8to1/mux8to1 (deflated 84%)
  adding: mux8to1/log_iter_4.txt (deflated 83%)
  adding: mux8to1/log_iter_2.txt (deflated 79%)
  adding: mux8to1/log_iter_0.txt (deflated 64%)
  adding: mux8to1/log.txt (deflated 86%)
  adding: mux8to1/mux8to1.v (deflated 70%)
  adding: mux8to1/log_iter_1.txt (deflated 73%)
  adding: mux8to1/log_iter_3.txt (deflated 83%)
  adding: verilog_project.zip (stored 0%)
  adding: half_adder/ (stored 0%)
  adding: half_adder/half_adder.v (deflated 63%)
  adding: half_adder/half_adder (deflated 75%)
  adding: half_adder/log_iter_0.txt (deflated 56%)
  adding: half_adder/log.txt (deflated 62%)
  adding: half_adder/log_iter_1.txt (deflated 63%)
  adding: half_adder/.ipynb_checkpoints/ (stored 0%)
  adding: mux8to1tb.v (deflated 56%)
  adding: adder4/ (stored 0%)
  adding: adder4/adder4.v (deflated 71%)
  adding: adder4/adder4 (deflated 87%)
  adding: adder4/log_iter_2.txt (deflated 78%)
  adding: adder4/log_iter_0.txt (deflat

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>